In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcomApp_MySQL_Read") \
    .master("spark://localhost:7077") \
    .config("spark.driver.host", "host.docker.internal")\
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.extraClassPath", "/opt/spark/jars/mysql-connector-j-8.3.0.jar") \
    .config("spark.driver.extraClassPath", "/opt/spark/jars/mysql-connector-j-8.3.0.jar") \
    .config("spark.executor.memory", "512m") \
    .config("spark.executor.cores", "1") \
    .config("spark.cores.max", "2") \
    .getOrCreate()

# Verify
spark.sparkContext._jvm.Class.forName("com.mysql.cj.jdbc.Driver")
print("✅ Driver loaded!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/18 17:11:59 WARN Utils: Your hostname, Abhisheks-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.113 instead (on interface en0)
26/05/18 17:11:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 17:11:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/18 17:11:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Driver loaded!


In [2]:
df = spark.createDataFrame([{'col1': ['abhi', 'abhishek']}])

In [3]:
df.show()

+----------------+
|            col1|
+----------------+
|[abhi, abhishek]|
+----------------+



In [4]:
try:
    df = spark.read \
        .format("jdbc") \
        .option("url", "jdbc:mysql://host.docker.internal:3306/ecom_oltp_db") \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .option("dbtable", "customers") \
        .option("user", "root") \
        .option("password", "password") \
        .load()
except Exception as e:
    print(str(e)) 

In [5]:
df.show()

+---+----------+-----------+-------------+--------------------+--------+-------------------+-------------------+
| id|address_id|       name|mobile_number|               email|  status|         created_at|         updated_at|
+---+----------+-----------+-------------+--------------------+--------+-------------------+-------------------+
|  1|     16896| Customer 1|   7475647151|customer1@example...|  active|2025-10-20 14:31:18|2025-10-20 14:31:18|
|  2|     21182| Customer 2|   6642130080|customer2@example...|  active|2025-09-13 07:36:18|2025-09-13 07:36:18|
|  3|     11768| Customer 3|   6119602969|customer3@example...|  active|2026-02-27 05:05:18|2026-02-27 05:05:18|
|  4|     20474| Customer 4|   7481412436|customer4@example...|  active|2025-07-10 17:55:18|2025-07-10 17:55:18|
|  5|     48035| Customer 5|   7147612961|customer5@example...|  active|2025-09-27 14:15:18|2025-09-27 14:15:18|
|  6|     13361| Customer 6|   6505766041|customer6@example...|  active|2026-05-15 10:26:18|2026

26/05/18 17:12:54 ERROR TaskSchedulerImpl: Lost executor 1 on 172.24.0.3: Worker shutting down
26/05/18 17:12:54 ERROR TaskSchedulerImpl: Lost executor 0 on 172.24.0.4: Worker shutting down
26/05/18 17:12:55 WARN StandaloneAppClient$ClientEndpoint: Connection to 172.24.0.2:7077 failed; waiting for master to reconnect...
26/05/18 17:12:55 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
